<a href="https://colab.research.google.com/github/WesleyVictors/projeto_desenvolve_de_dados/blob/main/gera%C3%A7%C3%A3o_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install mimesis

import pandas as pd
import numpy as np
from mimesis import Generic
from mimesis.locales import Locale
import random

g = Generic(locale=Locale.PT_BR)
num_rows = 5000

produtos_db = [
    {"prod": "MacBook Air M2", "marca": "Apple", "cat": "Notebooks", "preco": 8500.00},
    {"prod": "Dell XPS 13", "marca": "Dell", "cat": "Notebooks", "preco": 7200.00},
    {"prod": "Samsung Galaxy Tab S9", "marca": "Samsung", "cat": "Tablets", "preco": 4500.00},
    {"prod": "iPhone 15 Pro", "marca": "Apple", "cat": "Smartphones", "preco": 9000.00},
    {"prod": "Monitor LG UltraWide 34", "marca": "LG", "cat": "Monitores", "preco": 2800.00},
    {"prod": "Teclado Logitech MX", "marca": "Logitech", "cat": "Periféricos", "preco": 600.00}
]

data = []

for i in range(1, num_rows + 1):
    p = random.choice(produtos_db)
    qtde = random.randint(1, 3)

    # 1. Lógica de DATAS (Formatos: dd/mm/aaaa, dd-mm-aaaa, aaaa-mm-dd)
    dt_obj = g.datetime.date(start=2023, end=2025)
    sorteio_dt = random.random()
    if sorteio_dt < 0.6:
        data_str = dt_obj.strftime('%d/%m/%Y')
    elif sorteio_dt < 0.85:
        data_str = dt_obj.strftime('%d-%m-%Y')
    else:
        data_str = dt_obj.strftime('%Y-%m-%d')

    # 2. Lógica de HORAS (Formatos: HH:MM:SS, HH:MM, HH:MM AM/PM)
    hr_obj = g.datetime.time()
    sorteio_hr = random.random()
    if sorteio_hr < 0.6:
        hora_str = hr_obj.strftime('%H:%M:%S') # Padrão completo
    elif sorteio_hr < 0.85:
        hora_str = hr_obj.strftime('%H:%M')    # Sem segundos
    else:
        hora_str = hr_obj.strftime('%I:%M %p') # Formato 12h (AM/PM)

    row = {
        "pedido_id": i,
        "cliente_id": g.numeric.integer_number(start=1000, end=5000),
        "cliente": g.person.full_name(),
        "cpf": f"{g.numeric.integer_number(100,999)}.{g.numeric.integer_number(100,999)}."
               f"{g.numeric.integer_number(100,999)}-{g.numeric.integer_number(10,99)}",
        "produto_id": g.numeric.integer_number(start=100, end=999),
        "produto": p['prod'],
        "marca": p['marca'],
        "categoria": p['cat'],
        "quantidade": qtde,
        "valor_unitario": p['preco'],
        "frete": round(random.uniform(15, 150), 2),
        "cupom_desconto": random.choice([0, 0, 0, 10, 30]),
        "canal_marketing": random.choice(["Google Ads", "Instagram", "E-mail", "Orgânico"]),
        "cidade": g.address.city(),
        "estado": g.address.state(abbr=True),
        "rua": g.address.street_name(),
        "numero": g.address.street_number(),
        "bairro": g.person.last_name() + " Village",
        "data_compra": data_str,
        "hora_compra": hora_str,
        "metodo_pagamento": random.choice(["Cartão", "Pix", "Boleto"]),
        "status_pedido": random.choice(["Entregue", "Entregue", "Cancelado", "Enviado"])
    }

    # Cálculos financeiros
    row["valor_total"] = round((row["valor_unitario"] * row["quantidade"]) + row["frete"] - row["cupom_desconto"], 2)
    row["lucro"] = round(row["valor_total"] * random.uniform(0.15, 0.28), 2)

    data.append(row)

df = pd.DataFrame(data)

# --- Inconsistências  ---
# Duplicatas
df = pd.concat([df, df.sample(n=30)], ignore_index=True)

# Valores Nulos
df.loc[df.sample(frac=0.01).index, ['cpf', 'marca']] = np.nan


df.to_csv('ecom_data.csv', index=False, encoding='utf-8-sig')

print(f"Sucesso! Arquivo gerado com {len(df)} linhas.")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 45.5 MB/s eta 0:00:00
Sucesso! Arquivo gerado com 5030 linhas.


In [ ]:
#Formatação de datas e horas

df = pd.read_csv('ecom_data.csv')

#DATAS
df['data_compra_limpa'] = pd.to_datetime(
    df['data_compra'],
    dayfirst=True,
    format='mixed')

#HORAS
df['hora_compra_limpa'] = pd.to_datetime(
    df['hora_compra'],
    format='mixed'
).dt.time

print(df[['data_compra_limpa', 'data_compra', 'hora_compra_limpa', 'hora_compra']].head())

df = df.drop(columns=['data_compra', 'hora_compra'])
df = df.rename(columns={
    'data_compra_limpa': 'data_compra',
    'hora_compra_limpa': 'hora_compra'
})

  data_compra_limpa data_compra hora_compra_limpa hora_compra
0        2025-07-08  08/07/2025          22:00:38    22:00:38
1        2025-07-13  13/07/2025          20:04:00    08:04 PM
2        2025-02-09  09/02/2025          19:56:00    07:56 PM
3        2023-01-29  29/01/2023          02:55:06    02:55:06
4        2024-10-26  26/10/2024          11:31:51    11:31:51


In [ ]:
df['data_compra'].isna().sum()
df['hora_compra'].isna().sum()

np.int64(0)

In [ ]:

# Tratamento de dados inconsistentes
colunas_financeiras = ['valor_unitario', 'frete', 'cupom_desconto', 'valor_total', 'lucro']

#Conversão Direta para Float
for col in colunas_financeiras:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Tratamento de Nulos com a sua Lógica de Cupom (%)
# Total = ((Preço * Qtd) - Desconto %) + Frete
mask_nulo = df['valor_total'].isna()

# Fórmula de porcentagem de desconto
df.loc[mask_nulo, 'valor_total'] = (
    (df['valor_unitario'] * df['quantidade']) * (1 - df['cupom_desconto'].fillna(0) / 100)
) + df['frete'].fillna(0)


# Se o lucro estiver nulo, utiliza 20% do valor
df['lucro'] = df['lucro'].fillna(df['valor_total'] * 0.20)

# filtro de valores negativos
df.loc[df['valor_total'] < 0, 'valor_total'] = 0

# Arredondamento para 2 casas decimais
df[colunas_financeiras] = df[colunas_financeiras].round(2)


print(df[colunas_financeiras].head())

   valor_unitario   frete  cupom_desconto  valor_total    lucro
0          7200.0   87.84               0     14487.84  2365.28
1          2800.0  108.76              10      2898.76   792.32
2           600.0   71.09               0      1871.09   448.34
3          9000.0   84.12               0     27084.12  4180.29
4           600.0   99.83              10      1889.83   346.34


In [ ]:
#tratamento de nulos
nulos = df.isnull().sum()
print(nulos)

pedido_id            0
cliente_id           0
cliente              0
cpf                 50
produto_id           0
produto              0
marca               50
categoria            0
quantidade           0
valor_unitario       0
frete                0
cupom_desconto       0
canal_marketing      0
cidade               0
estado               0
rua                  0
numero               0
bairro               0
metodo_pagamento     0
status_pedido        0
valor_total          0
lucro                0
data_compra          0
hora_compra          0
dtype: int64


In [ ]:
df['marca'] = df['marca'].fillna('Marca não identificada')
df['cpf'] = df['cpf'].fillna('000.000.000-00')

#tratamento cpf
df['cpf'] = df['cpf'].str.replace(r'\D', '', regex=True)
#preenche com zeros à esquerda (zfill)
df['cpf'] = df['cpf'].astype(str).str.zfill(11)

print(df['cpf'].str.len().unique())

[11]


In [ ]:
#tratamento dos estados
df['estado'] = df['estado'].str.replace('BR-', '', regex=False).str.strip()

In [ ]:
# Tratamento de duplicados
linhas_duplicadas = df[df.duplicated(keep=False)]
print(linhas_duplicadas.head())

      pedido_id  cliente_id          cliente          cpf  produto_id  \
948         949        4905      Vitiza Lott  33881634587         317   
1323       1324        2459   Nanina Bottega  92976231755         959   
1377       1378        4097  Procópio Penido  71315718196         124   
1394       1395        1849  Grimanesa Eller  59753089068         523   
1398       1399        3888    Teotónio Boge  64645312081         749   

                      produto     marca    categoria  quantidade  \
948       Teclado Logitech MX  Logitech  Periféricos           1   
1323            iPhone 15 Pro     Apple  Smartphones           3   
1377  Monitor LG UltraWide 34        LG    Monitores           1   
1394              Dell XPS 13      Dell    Notebooks           3   
1398            iPhone 15 Pro     Apple  Smartphones           1   

      valor_unitario  ...  estado                     rua numero  \
948            600.0  ...      MS             Rua Vitoria   1111   
1323          90

In [ ]:
df_limpo = df.drop_duplicates(keep='first')

In [ ]:
df_limpo = df_limpo.reset_index(drop=True)
print(f"Total de linhas antes: {len(df)}")
print(f"Total de linhas depois: {len(df_limpo)}")

Total de linhas antes: 5030
Total de linhas depois: 5001


In [ ]:
display(df_limpo.head(8))

,pedido_id,cliente_id,cliente,cpf,produto_id,produto,marca,categoria,quantidade,valor_unitario,...,estado,rua,numero,bairro,metodo_pagamento,status_pedido,valor_total,lucro,data_compra,hora_compra
0,1,4421,Teliano Vadagnin,42354610031,375,Dell XPS 13,Dell,Notebooks,2,7200.0,...,CE,Rua Bela Vista,270,Pagani Village,Pix,Entregue,14487.84,2365.28,2025-07-08,22:00:38
1,2,3936,Leanor Bosio,50157050867,298,Monitor LG UltraWide 34,LG,Monitores,1,2800.0,...,PE,Túnel Engenheiro Marques Porto,586,e Almeida Village,Pix,Enviado,2898.76,792.32,2025-07-13,20:04:00
2,3,1559,Rolim Valani,41655135416,703,Teclado Logitech MX,Logitech,Periféricos,3,600.0,...,PR,Rua Abreu Jr.,364,Binelle Village,Boleto,Enviado,1871.09,448.34,2025-02-09,19:56:00
3,4,3479,Jacob Cosme,42235910416,826,iPhone 15 Pro,Apple,Smartphones,3,9000.0,...,RJ,Avenida Adeodato Pires,220,Oinhos Village,Boleto,Cancelado,27084.12,4180.29,2023-01-29,02:55:06
4,5,1241,Cirilo Woodward,76730864986,231,Teclado Logitech MX,Logitech,Periféricos,3,600.0,...,AC,Rua Conde Bernadotte,762,Fioravante Village,Boleto,Cancelado,1889.83,346.34,2024-10-26,11:31:51
5,6,3560,Vânio Dessaure,51722835399,369,Teclado Logitech MX,Logitech,Periféricos,1,600.0,...,RJ,Rua São José,572,Lupino Village,Boleto,Entregue,626.99,130.50,2024-09-20,13:33:35
6,7,1968,Guilhermino Meirelles,26723751841,973,Monitor LG UltraWide 34,LG,Monitores,2,2800.0,...,ES,Rua Cel. Emídio Piedade,1089,Buson Village,Cartão,Entregue,5657.11,1468.99,2023-08-04,03:53:21
7,8,1320,Seleso Toneto,25682051950,123,Dell XPS 13,Dell,Notebooks,1,7200.0,...,AP,Avenida Lucas Evangelista,353,Rizenente Village,Boleto,Entregue,7228.26,1395.01,2025-10-17,01:57:00


In [ ]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5001 entries, 0 to 5000
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   pedido_id         5001 non-null   int64         
 1   cliente_id        5001 non-null   int64         
 2   cliente           5001 non-null   object        
 3   cpf               5001 non-null   object        
 4   produto_id        5001 non-null   int64         
 5   produto           5001 non-null   object        
 6   marca             5001 non-null   object        
 7   categoria         5001 non-null   object        
 8   quantidade        5001 non-null   int64         
 9   valor_unitario    5001 non-null   float64       
 10  frete             5001 non-null   float64       
 11  cupom_desconto    5001 non-null   int64         
 12  canal_marketing   5001 non-null   object        
 13  cidade            5001 non-null   object        
 14  estado            5001 n

In [ ]:
#removendo anomalia na geração da cidade
mask_anomalous_cidade = df_limpo['cidade'].str.contains("INSERT INTO", na=False)
num_anomalous_cidades = mask_anomalous_cidade.sum()


df_limpo.loc[mask_anomalous_cidade, 'cidade'] = np.nan
df_limpo['cidade'] = df_limpo['cidade'].fillna('Cidade Desconhecida')
display(df_limpo[mask_anomalous_cidade].head())

,pedido_id,cliente_id,cliente,cpf,produto_id,produto,marca,categoria,quantidade,valor_unitario,...,estado,rua,numero,bairro,metodo_pagamento,status_pedido,valor_total,lucro,data_compra,hora_compra


In [ ]:
df_limpo.to_csv("ecommerce_data.csv", index=False, encoding="utf-8")